# Notebook 1：一室模型基础PK/PD模拟

本Notebook是临床药学 PK/PD 交互式模拟平台的第一个基础模块。

本notebook通过最经典的一室模型，帮助大家建立临床药学中 PK/PD 的基本思维：

- 给药剂量如何影响血药浓度？
- 静脉给药和口服给药的浓度–时间曲线有什么不同？
- 分布容积 Vd 和清除率 CL 如何影响血药浓度？
- 半衰期 t1/2 和 AUC 如何计算？
- 血药浓度如何进一步转化为药物效应？
- 为什么剂量增加不一定意味着临床获益增加？

本Notebook的核心逻辑是：

$$
Dose \rightarrow Concentration(t) \rightarrow Exposure \rightarrow Effect
$$

也就是说，药物剂量并不直接等于药物效应。  
剂量首先决定体内药物浓度，浓度再通过药效学模型转化为药物效应。

## 1. 学习目标

完成本notebook后，你应该能够：

1. 解释一室模型中的核心 PK 参数，包括剂量、表观分布容积、清除率、消除速率常数和半衰期。
2. 比较静脉推注和口服给药后的血药浓度-时间曲线特征。
3. 描述生物利用度、吸收速率、分布容积和清除率如何影响 $C_{max}$、$T_{max}$、AUC 和半衰期。
4. 利用 MEC 和 MTC 初步判断疗效不足和毒性风险。
5. 使用基础 Emax 模型理解血药浓度与药物效应之间的关系。

## 2. 一室模型的基本概念

一室模型是最简单、最基础的药代动力学模型。

它将人体简化为一个“混合均匀的容器”。药物进入体内后，假设迅速分布到一个表观空间中，这个空间的大小称为表观分布容积：

$$
V_d
$$

药物从体内被消除的能力称为清除率：

$$
CL
$$

消除速率常数为：

$$
k = \frac{CL}{V_d}
$$

半衰期为：

$$
t_{1/2} = \frac{0.693}{k}
$$

也可以写成：

$$
t_{1/2} = \frac{0.693 \times V_d}{CL}
$$

因此，半衰期不是一个孤立参数，而是由 Vd 和 CL 共同决定。

## 3. 静脉快速给药的一室模型

对于静脉快速给药，药物直接进入体循环，不存在吸收过程。

给药后血药浓度可表示为：

$$
C(t) = \frac{Dose}{V_d} \cdot e^{-kt}
$$

其中：

$$
k = \frac{CL}{V_d}
$$

静脉快速给药后，最高浓度通常出现在给药后即刻：

$$
C_{max} = C_0 = \frac{Dose}{V_d}
$$

单次静脉给药后的 AUC 为：

$$
AUC = \frac{Dose}{CL}
$$

变量含义：

| 符号 | 含义 | 常用单位 |
|---|---|---|
| Dose | 给药剂量 | mg |
| Vd | 表观分布容积 | L |
| CL | 清除率 | L/h |
| k | 消除速率常数 | 1/h |
| t1/2 | 半衰期 | h |
| C(t) | t 时刻血药浓度 | mg/L |
| Cmax | 峰浓度 | mg/L |
| AUC | 药时曲线下面积 | mg·h/L |

注意：

- Dose 越大，初始血药浓度越高。
- Vd 越大，同样剂量被“稀释”得越多，初始浓度越低。
- CL 越大，药物清除越快，血药浓度下降越快。
- k 由 CL 和 Vd 共同决定。
- AUC 与 Dose 成正比，与 CL 成反比。

## 4. 口服给药的一室模型

口服给药与静脉给药不同。药物需要先从胃肠道吸收入体循环，因此多了两个重要参数：

| 参数 | 含义 |
|---|---|
| F | 生物利用度，表示进入体循环的比例 |
| ka | 吸收速率常数，表示吸收快慢 |

对于一级吸收、一级消除的一室模型，口服给药后的血药浓度可表示为：

$$
C(t) = \frac{F \cdot Dose \cdot k_a}{V_d(k_a-k)}
\left(e^{-kt} - e^{-k_a t}\right)
$$

其中：

$$
k = \frac{CL}{V_d}
$$

口服给药后的 AUC 为：

$$
AUC = \frac{F \cdot Dose}{CL}
$$

与静脉给药相比，口服给药有以下特点：

- 浓度不是从最高点开始下降，而是先上升后下降。
- 存在吸收过程，因此有 Tmax。
- Cmax 受 Dose、F、Vd、CL 和 ka 共同影响。
- F 降低会使全身暴露 AUC 降低。
- ka 越大，吸收越快，Tmax 通常越早。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


def one_compartment_iv_bolus(t, dose_mg, vd_l, cl_l_h):
    """
    One-compartment IV bolus model.
    """
    k_elim = cl_l_h / vd_l
    concentration = (dose_mg / vd_l) * np.exp(-k_elim * t)
    auc = dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def one_compartment_oral(t, dose_mg, vd_l, cl_l_h, ka_h, bioavailability):
    """
    One-compartment oral dosing model with first-order absorption and elimination.
    """
    k_elim = cl_l_h / vd_l

    if np.isclose(ka_h, k_elim):
        concentration = (
            bioavailability
            * dose_mg
            / vd_l
            * k_elim
            * t
            * np.exp(-k_elim * t)
        )
    else:
        concentration = (
            bioavailability
            * dose_mg
            * ka_h
            / (vd_l * (ka_h - k_elim))
            * (np.exp(-k_elim * t) - np.exp(-ka_h * t))
        )

    concentration = np.maximum(concentration, 0)
    auc = bioavailability * dose_mg / cl_l_h
    half_life = np.log(2) / k_elim

    return concentration, k_elim, half_life, auc


def calculate_pk_metrics(t, concentration, auc, half_life):
    """
    Calculate basic PK metrics.
    """
    cmax = np.max(concentration)
    tmax = t[np.argmax(concentration)]

    metrics = {
        "Cmax": cmax,
        "Tmax": tmax,
        "AUC": auc,
        "Half-life": half_life
    }

    return metrics

## 5. 交互模拟 1：比较静脉给药和口服给药

下面的模拟可以选择两种给药途径：

- IV bolus：静脉快速给药
- Oral：口服给药

请观察：

- 静脉给药后浓度是否从最高点开始下降？
- 口服给药后浓度是否先上升再下降？
- F 改变时，口服给药的 AUC 如何变化？
- ka 改变时，Tmax 和 Cmax 如何变化？
- CL 改变时，消除速度和 AUC 如何变化？

In [ ]:
def plot_basic_pk_model(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    if route == "IV bolus":
        concentration, k_elim, half_life, auc = one_compartment_iv_bolus(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )
    else:
        concentration, k_elim, half_life, auc = one_compartment_oral(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    metrics = calculate_pk_metrics(
        t=t,
        concentration=concentration,
        auc=auc,
        half_life=half_life
    )

    fig, ax = plt.subplots()
    ax.plot(t, concentration, linewidth=2, label=f"{route}")

    ax.scatter(metrics["Tmax"], metrics["Cmax"], zorder=5)
    ax.text(
        metrics["Tmax"],
        metrics["Cmax"],
        f"  Cmax={metrics['Cmax']:.2f}, Tmax={metrics['Tmax']:.2f} h",
        va="center"
    )

    ax.set_title("One-Compartment PK Model")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Parameter": [
            "Route",
            "Dose",
            "Vd",
            "CL",
            "ka",
            "Bioavailability",
            "Elimination rate constant",
            "Half-life",
            "Cmax",
            "Tmax",
            "AUC"
        ],
        "Value": [
            route,
            f"{dose_mg:.1f} mg",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.1f} L/h",
            f"{ka_h:.2f} 1/h",
            f"{bioavailability:.2f}",
            f"{k_elim:.4f} 1/h",
            f"{metrics['Half-life']:.2f} h",
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Tmax']:.2f} h",
            f"{metrics['AUC']:.2f} mg*h/L"
        ]
    })

    display(summary)


interact(
    plot_basic_pk_model,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 6. 观察任务 1：给药途径对浓度曲线的影响

请完成以下操作：

### 任务 A：静脉给药

设置：

- Route = IV bolus
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- Time = 24 h

观察：

- 浓度是否从最高点开始下降？
- Tmax 是否接近 0？
- Cmax 是否约等于 Dose / Vd？
- AUC 是否等于 Dose / CL？

### 任务 B：口服给药

设置：

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- ka = 1.2 1/h
- F = 0.8
- Time = 24 h

观察：

- 浓度是否先上升后下降？
- Tmax 是否大于 0？
- AUC 是否低于同剂量静脉给药？
- 为什么口服给药的 AUC 与 F 有关？

### 任务 C：改变吸收速率

保持其他参数不变，将 ka 分别设置为：

- 0.3 1/h
- 1.2 1/h
- 3.0 1/h

观察：

- ka 增大时，Tmax 是提前还是延后？
- ka 增大时，Cmax 是否通常升高？
- ka 很小时，浓度曲线是否更平缓？

## 7. 半衰期和 AUC 的临床意义

半衰期表示血药浓度下降一半所需的时间。

对于一级消除过程：

- 经过 1 个半衰期，浓度约剩余 50%
- 经过 2 个半衰期，浓度约剩余 25%
- 经过 3 个半衰期，浓度约剩余 12.5%
- 经过 4 个半衰期，浓度约剩余 6.25%
- 经过 5 个半衰期，浓度约剩余 3.125%

AUC 表示药物暴露量。

对于静脉给药：

$$
AUC = \frac{Dose}{CL}
$$

对于口服给药：

$$
AUC = \frac{F \cdot Dose}{CL}
$$

因此：

- Dose 增加，AUC 增加。
- CL 增加，AUC 降低。
- 口服给药时，F 降低，AUC 降低。
- 肾功能或肝功能下降时，如果 CL 下降，AUC 可能升高，毒性风险可能增加。

In [ ]:
def plot_iv_and_oral_comparison(
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    conc_iv, k_elim, half_life_iv, auc_iv = one_compartment_iv_bolus(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h
    )

    conc_oral, _, half_life_oral, auc_oral = one_compartment_oral(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        ka_h=ka_h,
        bioavailability=bioavailability
    )

    metrics_iv = calculate_pk_metrics(t, conc_iv, auc_iv, half_life_iv)
    metrics_oral = calculate_pk_metrics(t, conc_oral, auc_oral, half_life_oral)

    fig, ax = plt.subplots()
    ax.plot(t, conc_iv, linewidth=2, label="IV bolus")
    ax.plot(t, conc_oral, linewidth=2, label="Oral")

    ax.set_title("IV Bolus vs Oral Dosing")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": ["Cmax", "Tmax", "AUC", "Half-life"],
        "IV bolus": [
            f"{metrics_iv['Cmax']:.2f} mg/L",
            f"{metrics_iv['Tmax']:.2f} h",
            f"{metrics_iv['AUC']:.2f} mg*h/L",
            f"{metrics_iv['Half-life']:.2f} h"
        ],
        "Oral": [
            f"{metrics_oral['Cmax']:.2f} mg/L",
            f"{metrics_oral['Tmax']:.2f} h",
            f"{metrics_oral['AUC']:.2f} mg*h/L",
            f"{metrics_oral['Half-life']:.2f} h"
        ]
    })

    display(summary)


interact(
    plot_iv_and_oral_comparison,
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(FloatSlider(value=500.0, description='Dose', max=2000.0, min=100.0, step=100.0), FloatSl…

## 8. 观察任务 2：静脉与口服给药比较

请在上面的图中比较同一剂量下的两条曲线。

思考：

1. 为什么静脉给药没有吸收相？
2. 为什么口服给药的 Tmax 晚于静脉给药？
3. 为什么口服给药的 AUC 通常低于静脉给药？
4. 如果 F = 1，口服给药和静脉给药的 AUC 是否相同？
5. 如果 ka 很小，口服曲线会出现什么变化？
6. 如果 CL 降低，静脉和口服给药的 AUC 会如何变化？

## 9. 治疗窗：从浓度到临床判断

在临床药学中，我们不仅关心血药浓度是多少，还关心浓度是否处于合理范围。

可以用两个简化概念帮助理解：

| 概念 | 含义 |
|---|---|
| MEC | Minimum Effective Concentration，最小有效浓度 |
| MTC | Minimum Toxic Concentration，最小毒性浓度 |

当血药浓度低于 MEC 时，可能疗效不足。

当血药浓度高于 MTC 时，可能毒性风险增加。

当血药浓度位于 MEC 和 MTC 之间时，可以认为处于治疗窗范围内：

$$
MEC \leq C(t) \leq MTC
$$

需要注意，本 Notebook 中的 MEC 和 MTC 仅用于教学模拟。真实临床判断还需要结合药物种类、适应证、患者器官功能、合并用药、病原体敏感性和治疗反应。

In [ ]:
def calculate_time_in_ranges(t, concentration, mec, mtc):
    """
    Calculate time below MEC, within therapeutic window, and above MTC.
    """
    dt = t[1] - t[0]

    time_below_mec = np.sum(concentration < mec) * dt
    time_within_window = np.sum(
        (concentration >= mec) & (concentration <= mtc)
    ) * dt
    time_above_mtc = np.sum(concentration > mtc) * dt

    return time_below_mec, time_within_window, time_above_mtc


def plot_therapeutic_window(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    mec=2,
    mtc=12,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    if route == "IV bolus":
        concentration, k_elim, half_life, auc = one_compartment_iv_bolus(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )
    else:
        concentration, k_elim, half_life, auc = one_compartment_oral(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    metrics = calculate_pk_metrics(t, concentration, auc, half_life)

    time_below_mec, time_within_window, time_above_mtc = calculate_time_in_ranges(
        t=t,
        concentration=concentration,
        mec=mec,
        mtc=mtc
    )

    fig, ax = plt.subplots()
    ax.plot(t, concentration, linewidth=2, label=f"{route}")
    ax.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax.fill_between(t, mec, mtc, alpha=0.15, label="Therapeutic window")

    ax.set_title("Concentration and Therapeutic Window")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Cmax",
            "Tmax",
            "AUC",
            "Half-life",
            "Time below MEC",
            "Time within therapeutic window",
            "Time above MTC"
        ],
        "Value": [
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Tmax']:.2f} h",
            f"{metrics['AUC']:.2f} mg*h/L",
            f"{metrics['Half-life']:.2f} h",
            f"{time_below_mec:.2f} h",
            f"{time_within_window:.2f} h",
            f"{time_above_mtc:.2f} h"
        ]
    })

    display(summary)


interact(
    plot_therapeutic_window,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=2, min=0.5, max=10, step=0.5, description="MEC"),
    mtc=FloatSlider(value=12, min=5, max=30, step=1, description="MTC"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 10. 观察任务 3：治疗窗与疗效/毒性风险

请完成以下操作：

### 任务 A：标准口服给药

设置：

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- ka = 1.2 1/h
- F = 0.8
- MEC = 2 mg/L
- MTC = 12 mg/L

记录：

- Cmax
- Tmax
- AUC
- Time below MEC
- Time within therapeutic window
- Time above MTC

### 任务 B：增加剂量

将 Dose 改为 1000 mg。

观察：

- Cmax 是否升高？
- AUC 是否升高？
- Time above MTC 是否增加？
- 增加剂量是否一定更安全？

### 任务 C：降低清除率

将 CL 改为 2 L/h。

观察：

- AUC 是否升高？
- 半衰期是否延长？
- Time above MTC 是否增加？
- 这种情况可以模拟哪些临床患者？

提示：肾功能下降、肝功能下降或药物相互作用均可能导致清除率下降。

## 11. 从 PK 到 PD：Emax 模型

PK 关注的是：

> 身体如何处理药物？

PD 关注的是：

> 药物如何产生效应？

在临床药学中，我们不仅要知道血药浓度是多少，还需要理解这个浓度能产生多大药效。

本 Notebook 使用最常见的 Emax 模型描述浓度和效应之间的关系：

$$
Effect = E_0 + \frac{E_{max} \cdot C^\gamma}{EC_{50}^{\gamma} + C^\gamma}
$$

其中：

| 参数 | 含义 |
|---|---|
| E0 | 基线效应 |
| Emax | 最大药效 |
| EC50 | 达到 50% 最大效应时的浓度 |
| gamma | Hill 系数，决定曲线陡峭程度 |
| C | 血药浓度 |

当浓度较低时，浓度增加可能带来明显药效增加。

当浓度较高时，效应逐渐接近平台。此时继续增加剂量，疗效增加可能有限，但毒性风险可能继续增加。

In [ ]:
def emax_effect(concentration, e0, emax, ec50, gamma):
    """
    Emax pharmacodynamic model.
    """
    concentration = np.asarray(concentration)

    effect = e0 + (
        emax * concentration**gamma
    ) / (
        ec50**gamma + concentration**gamma
    )

    return effect


def plot_emax_curve(
    e0=0,
    emax=100,
    ec50=5,
    gamma=1.5,
    c_max=30
):
    c = np.linspace(0, c_max, 500)

    effect = emax_effect(
        concentration=c,
        e0=e0,
        emax=emax,
        ec50=ec50,
        gamma=gamma
    )

    effect_at_ec50 = emax_effect(
        concentration=ec50,
        e0=e0,
        emax=emax,
        ec50=ec50,
        gamma=gamma
    )

    fig, ax = plt.subplots()
    ax.plot(c, effect, linewidth=2, label="Effect")
    ax.scatter(ec50, effect_at_ec50, zorder=5)
    ax.axvline(ec50, linestyle="--", label=f"EC50 = {ec50:.2f} mg/L")
    ax.axhline(effect_at_ec50, linestyle=":", label="Effect at EC50")

    ax.set_title("Emax Concentration-Effect Relationship")
    ax.set_xlabel("Concentration (mg/L)")
    ax.set_ylabel("Effect (%)")
    ax.set_ylim(0, e0 + emax * 1.1)
    ax.legend()
    plt.show()


interact(
    plot_emax_curve,
    e0=FloatSlider(value=0, min=0, max=50, step=5, description="E0"),
    emax=FloatSlider(value=100, min=20, max=150, step=10, description="Emax"),
    ec50=FloatSlider(value=5, min=0.5, max=20, step=0.5, description="EC50"),
    gamma=FloatSlider(value=1.5, min=0.5, max=5, step=0.5, description="Hill"),
    c_max=FloatSlider(value=30, min=5, max=80, step=5, description="C max")
);

interactive(children=(FloatSlider(value=0.0, description='E0', max=50.0, step=5.0), FloatSlider(value=100.0, d…

## 12. 观察任务 4：PD 参数对药效曲线的影响

请完成以下操作：

### 任务 A：改变 EC50

设置：

- Emax = 100
- EC50 = 5 mg/L
- Hill = 1.5

然后将 EC50 改为 10 mg/L。

观察：

- 曲线是否右移？
- 同样浓度下药效是否降低？
- EC50 增大代表药物敏感性增强还是降低？

### 任务 B：改变 Hill 系数

将 Hill 从 1.5 改为 4。

观察：

- 曲线是否更陡？
- 浓度小幅变化是否可能导致效应明显变化？
- 这类药物在剂量调整时是否可能需要更谨慎？

### 任务 C：改变 Emax

将 Emax 从 100 改为 60。

观察：

- 最大效应是否下降？
- 即使浓度很高，效应是否仍然无法超过新的 Emax？

## 13. 综合模拟：从给药剂量到血药浓度，再到药物效应

现在我们把 PK 和 PD 连接起来。

完整过程为：

$$
Dose \rightarrow C(t) \rightarrow Effect(t)
$$

该模拟同时显示：

- 血药浓度–时间曲线
- 治疗窗
- 药物效应–时间曲线

你可以选择静脉给药或口服给药，并调整 PK 和 PD 参数。

In [ ]:
def plot_integrated_pkpd(
    route="Oral",
    dose_mg=500,
    vd_l=50,
    cl_l_h=5,
    ka_h=1.2,
    bioavailability=0.8,
    mec=2,
    mtc=12,
    emax=100,
    ec50=5,
    gamma=1.5,
    t_end_h=24
):
    t = np.linspace(0, t_end_h, 1000)

    if route == "IV bolus":
        concentration, k_elim, half_life, auc = one_compartment_iv_bolus(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h
        )
    else:
        concentration, k_elim, half_life, auc = one_compartment_oral(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            ka_h=ka_h,
            bioavailability=bioavailability
        )

    effect = emax_effect(
        concentration=concentration,
        e0=0,
        emax=emax,
        ec50=ec50,
        gamma=gamma
    )

    metrics = calculate_pk_metrics(t, concentration, auc, half_life)

    time_below_mec, time_within_window, time_above_mtc = calculate_time_in_ranges(
        t=t,
        concentration=concentration,
        mec=mec,
        mtc=mtc
    )

    max_effect = np.max(effect)
    time_max_effect = t[np.argmax(effect)]
    average_effect = np.trapz(effect, t) / (t[-1] - t[0])

    fig, ax1 = plt.subplots(figsize=(10, 6))

    ax1.plot(t, concentration, linewidth=2, label="Concentration")
    ax1.axhline(mec, linestyle="--", label=f"MEC = {mec:.1f} mg/L")
    ax1.axhline(mtc, linestyle="--", label=f"MTC = {mtc:.1f} mg/L")
    ax1.fill_between(t, mec, mtc, alpha=0.15, label="Therapeutic window")

    ax1.set_xlabel("Time (h)")
    ax1.set_ylabel("Concentration (mg/L)")

    ax2 = ax1.twinx()
    ax2.plot(t, effect, linewidth=2, linestyle="-.", label="Effect")
    ax2.set_ylabel("Effect (%)")

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

    ax1.set_title("Integrated PK/PD Simulation")
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Route",
            "Cmax",
            "Tmax",
            "AUC",
            "Half-life",
            "Maximum effect",
            "Time of maximum effect",
            "Average effect",
            "Time below MEC",
            "Time within therapeutic window",
            "Time above MTC"
        ],
        "Value": [
            route,
            f"{metrics['Cmax']:.2f} mg/L",
            f"{metrics['Tmax']:.2f} h",
            f"{metrics['AUC']:.2f} mg*h/L",
            f"{metrics['Half-life']:.2f} h",
            f"{max_effect:.2f}%",
            f"{time_max_effect:.2f} h",
            f"{average_effect:.2f}%",
            f"{time_below_mec:.2f} h",
            f"{time_within_window:.2f} h",
            f"{time_above_mtc:.2f} h"
        ]
    })

    display(summary)


interact(
    plot_integrated_pkpd,
    route=Dropdown(
        options=["IV bolus", "Oral"],
        value="Oral",
        description="Route"
    ),
    dose_mg=FloatSlider(value=500, min=100, max=2000, step=100, description="Dose"),
    vd_l=FloatSlider(value=50, min=10, max=150, step=5, description="Vd"),
    cl_l_h=FloatSlider(value=5, min=1, max=20, step=1, description="CL"),
    ka_h=FloatSlider(value=1.2, min=0.1, max=5, step=0.1, description="ka"),
    bioavailability=FloatSlider(value=0.8, min=0.1, max=1.0, step=0.05, description="F"),
    mec=FloatSlider(value=2, min=0.5, max=10, step=0.5, description="MEC"),
    mtc=FloatSlider(value=12, min=5, max=30, step=1, description="MTC"),
    emax=FloatSlider(value=100, min=20, max=150, step=10, description="Emax"),
    ec50=FloatSlider(value=5, min=0.5, max=20, step=0.5, description="EC50"),
    gamma=FloatSlider(value=1.5, min=0.5, max=5, step=0.5, description="Hill"),
    t_end_h=FloatSlider(value=24, min=6, max=72, step=6, description="Time")
);

interactive(children=(Dropdown(description='Route', index=1, options=('IV bolus', 'Oral'), value='Oral'), Floa…

## 14. 观察任务 5：完整 PK/PD 判断

请使用综合模拟完成以下任务。

### 任务 A：标准口服给药

设置：

- Route = Oral
- Dose = 500 mg
- Vd = 50 L
- CL = 5 L/h
- ka = 1.2 1/h
- F = 0.8
- MEC = 2 mg/L
- MTC = 12 mg/L
- Emax = 100
- EC50 = 5 mg/L
- Hill = 1.5

记录：

- Cmax
- Tmax
- AUC
- Half-life
- Maximum effect
- Average effect
- Time within therapeutic window

### 任务 B：口服吸收变慢

只将 ka 改为 0.3 1/h。

观察：

- Tmax 是否延后？
- Cmax 是否降低？
- AUC 是否明显改变？
- 药物效应出现是否延迟？

思考：

> 如果一个缓释制剂吸收更慢，它的 Cmax、Tmax 和效应持续时间可能如何变化？

### 任务 C：生物利用度下降

只将 F 改为 0.4。

观察：

- Cmax 是否降低？
- AUC 是否降低？
- Average effect 是否降低？
- Time below MEC 是否增加？

思考：

> 如果患者吸收不良，或者食物/药物相互作用降低了口服生物利用度，可能会出现什么治疗问题？

### 任务 D：清除率下降

将 F 恢复为 0.8，只将 CL 改为 2 L/h。

观察：

- AUC 是否升高？
- Half-life 是否延长？
- Time above MTC 是否增加？
- 是否提示毒性风险增加？

思考：

> 对肾功能下降或肝功能下降患者，为什么即使剂量不变，也可能需要调整给药方案？

## 15. 自测题：剂量、浓度与药效

请根据本 Notebook 的内容完成以下自测题。建议先独立作答，再查看下一单元格中的参考答案。

---

### 题目 1：剂量增加一定会带来更好的临床获益吗？

某药物采用一室模型描述，增加 Dose 后，血药浓度升高。以下说法哪一项最合理？

A. 剂量越大，疗效一定越好，且安全性不变  
B. 剂量增加可能提高疗效，但也可能增加毒性风险  
C. 只要 Cmax 升高，AUC 一定降低  
D. 只要药效接近 Emax，继续增加剂量仍会带来等比例疗效增加  

---

### 题目 2：口服给药中，生物利用度 F 降低最直接影响什么？

A. 增加进入体循环的药物比例  
B. 降低口服给药后的 AUC  
C. 使药物清除率 CL 一定升高  
D. 使半衰期一定缩短  

---

### 题目 3：清除率 CL 降低时，以下哪种变化最可能发生？

A. AUC 降低  
B. 半衰期缩短  
C. 血药浓度下降更快  
D. AUC 升高，半衰期延长  

---

### 题目 4：关于 Emax 模型，以下说法哪一项正确？

A. 浓度和效应永远呈线性关系  
B. EC50 越大，通常表示达到相同效应需要更高浓度  
C. Emax 越小，药物最大效应越高  
D. Hill 系数与浓度-效应曲线形状无关  

---

### 题目 5：口服吸收速率常数 ka 变小时，最可能出现什么现象？

A. Tmax 延后，浓度上升更慢  
B. Tmax 提前，Cmax 一定显著升高  
C. 生物利用度 F 一定变为 1  
D. 清除率 CL 一定降低  

## 16. 自测题参考答案

### 题目 1

**参考答案：B**

**解析：**  
剂量增加通常会使血药浓度和 AUC 升高。如果原浓度低于有效范围，增加剂量可能提高疗效；但如果浓度超过 MTC，则可能增加毒性风险。根据 Emax 模型，当效应接近 Emax 后，继续增加浓度带来的额外药效可能有限。

---

### 题目 2

**参考答案：B**

**解析：**  
口服给药后：

$$
AUC = \frac{F \cdot Dose}{CL}
$$

在 Dose 和 CL 不变时，F 降低会使进入体循环的药量减少，因此 AUC 降低。F 本身不直接决定 CL 或半衰期。

---

### 题目 3

**参考答案：D**

**解析：**  
一室模型中：

$$
k = \frac{CL}{V_d}
$$

$$
t_{1/2} = \frac{0.693 \times V_d}{CL}
$$

当 CL 降低时，消除速率常数 k 降低，药物消除变慢，半衰期延长。同时：

$$
AUC = \frac{Dose}{CL}
$$

口服给药时：

$$
AUC = \frac{F \cdot Dose}{CL}
$$

因此 CL 降低会导致 AUC 升高。

---

### 题目 4

**参考答案：B**

**解析：**  
Emax 模型描述的是非线性的浓度-效应关系：

$$
Effect = E_0 + \frac{E_{max} \cdot C^\gamma}{EC_{50}^{\gamma} + C^\gamma}
$$

EC50 表示达到 50% 最大效应所需的浓度。EC50 越大，通常说明需要更高浓度才能达到相同效应。Emax 决定最大效应，Hill 系数影响曲线陡峭程度。

---

### 题目 5

**参考答案：A**

**解析：**  
ka 表示口服药物的吸收速度。ka 较小时，药物吸收较慢，浓度上升更慢，Tmax 通常延后，Cmax 可能降低或曲线更平缓。ka 不等同于 F，也不直接决定 CL。

## 本节总结

本 notebook 介绍了一室模型中最基础的 PK/PD 概念。

核心要点包括：

1. 静脉推注给药通常在给药后立即达到最高浓度，随后按指数规律下降。
2. 口服给药同时包含吸收和消除过程，因此浓度通常先升高后降低。
3. 生物利用度、吸收速率、分布容积和清除率共同决定浓度-时间曲线、$C_{max}$、$T_{max}$、AUC 和半衰期。
4. MEC 和 MTC 可以作为初步判断疗效不足和毒性风险的简化框架。
5. Emax 模型说明药物效应通常随浓度升高而增加，但这种增加往往是非线性的。
6. 剂量优化的基本目标是在疗效和安全性之间取得平衡。

本节的完整逻辑可以概括为：

$$
Route + Dose \rightarrow Concentration(t) \rightarrow Exposure \rightarrow Effect \rightarrow Efficacy/Safety
$$

下一节将进一步学习：

> 多剂量给药、稳态浓度和 PK/PD 关系。